## 获取任务sql

In [1]:
from voydstools.common.DataServiceAPI import DataServiceHttp

In [2]:
# For test
x_app_key = '50214295'
secret_key = 'e467683796fe3ae3792bf83fa0ef2796bb47dd90'


def get_sql_from_task_id(project_id, task_id):
    api = DataServiceHttp(x_app_key, secret_key)
    api.set_api("get_tabel_sql_temp_sql_df")
    query_data = {
        "dt": "2024-09-13",
        "project_id": project_id,
        "task_id": task_id
    }
    response = api.post(query_data)
    return response[0]['file_content']


In [3]:
sql = get_sql_from_task_id('3100', '31041901')
sql

请求：http://10.88.128.15:8000/dataservice/gateway/v1/api/get_tabel_sql_temp_sql_df，数据：{'dt': '2024-09-13', 'project_id': '3100', 'task_id': '31041901', 'pageSize': 5000, 'page': 1}


"--@exclude_dependency=sparklingwater.gateway_daily_hudi_ods_brp_--SPARK_SQL_brp_--********************************************************************--_brp_--author:zhengzong_brp_--create time:2024-08-01 16:26:40_brp_--desc:雨刷器档位_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_--- DROP TABLE IF EXISTS app_trip_rt_wipe;_brp_-- create table if not exists app_trip_rt_wiper_brp_-- (_brp_--     rigel_meta_trip_id string_brp_--     ,wiper_speed double_brp_--     ,wiper_speed_cn string_brp_-- )_brp_-- partitioned by_brp_-- (_brp_--     rigel_meta_event_date string comment '分区'_brp_-- )_brp_-- ;_brp__brp__brp_-- insert overwrite table app_trip_rt_wiper partition (rigel_meta_event_date = '${BIZ_DATE_LINE}')_brp_-- select_brp_--     rigel_meta_trip_id,_brp_--     max(vehicle_detail_info__wiper_speed) as wiper_speed,_brp_--     case   _brp_--         when sum(case when vehicle_detail_info__wiper_speed in (5,6) then 1 else 0 end) > 3

## 字段血缘关系

In [4]:
from sqllineage.runner import LineageRunner
# SUPPORTED_DIALECTS = list(dialect.label for dialect in dialect_readout())
# SUPPORTED_DIALECTS

In [5]:
sql = """
CREATE TABLE if not exists  `dim_rt_trip_odd_info_df`(
  `trip_id` string COMMENT 'trip_id', 
  `trip_odd` string COMMENT 'trip_odd',
  `create_date` string COMMENT 'create_date'
  )
PARTITIONED BY ( 
  `dt` string)
;

WITH app_rt_trip_issue_detail_hf AS (
    SELECT DISTINCT
        trip_id,
        region,
        create_date
    FROM 
        vgds.app_rt_trip_issue_detail_hf
    WHERE 
        dt = '2024-09-11-15'
),

trip_odd_data AS (
    SELECT DISTINCT
        b.trip_id, 
        CASE
            WHEN b.create_date <= a.create_static_date THEN a.trip_odd 
            ELSE 
                CASE
                    WHEN b.region like '%guangzhou%' or b.region like '%广州%' THEN 'ODD2'
                    WHEN b.region like '%beijing_yizhuang%' or b.region like '%北京亦庄%' THEN 'ODD2'
                    WHEN b.region like '%shanghai%' or b.region like '%上海%' THEN 'ODD3'
                    ELSE 'ODD-OTHER'
                END
        END AS trip_odd,
        b.create_date
    FROM 
        vgds.dim_rt_trip_odd_static_df a
    FULL OUTER JOIN 
        app_rt_trip_issue_detail_hf b
    ON a.trip_id = b.trip_id
)

INSERT OVERWRITE TABLE dim_rt_trip_odd_info_df partition (dt = '2024-09-11-15')
select 
    trip_id
    ,trip_odd s
    ,create_date
from
    trip_odd_data;
"""


In [6]:
from sqllineage.runner import LineageRunner
result = LineageRunner(sql,dialect='non-validating')
print(result)

Statements(#): 2
Source Tables:
    vgds.app_rt_trip_issue_detail_hf
    vgds.dim_rt_trip_odd_static_df
Target Tables:
    vgds.dim_rt_trip_odd_info_df



/tmp/ipykernel_13483/2817020059.py:2: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(sql,dialect='non-validating')


In [7]:
# result.draw()


## vgds库sql代码查询

In [8]:
sql = get_sql_from_task_id('3100', '31041901')

请求：http://10.88.128.15:8000/dataservice/gateway/v1/api/get_tabel_sql_temp_sql_df，数据：{'dt': '2024-09-13', 'project_id': '3100', 'task_id': '31041901', 'pageSize': 5000, 'page': 1}


In [10]:
def format_sql(sql_str):
    # 替换多个空格为单个空格，避免冗余
    formatted_sql = ' '.join(sql_str.split())

    # 按照指定的符号、关键词等进行换行和格式化
    formatted_sql = formatted_sql.replace('_brp__brp_', '\n\n') \
                                 .replace('_brp_ _', '\n,') \
                                 .replace('__brp_', ',\n') \
                                 .replace('_brp_', '\n') \
                                 .replace('_brp', '\n') \
                                #  .replace('_ ', ',') \

    # 去掉可能多余的空格
    formatted_sql = formatted_sql.replace(' ;', ';')
    # 如果最后不是分号结尾，加上分号
    if formatted_sql[-1] != ';':
        formatted_sql += ';'

    return formatted_sql
    

In [11]:
formatted_sql = format_sql(sql)

result = LineageRunner(formatted_sql,dialect='non-validating')
print(result)

Statements(#): 2
Source Tables:
    sparklingwater.gateway_daily_hudi_ods
Target Tables:
    vgds.app_trip_rt_wiper_df_all



/tmp/ipykernel_13483/887143189.py:3: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(formatted_sql,dialect='non-validating')


In [12]:
import os

def get_source_tables_from_sql(sql,format=False):
    try:
        if format:
            formatted_sql = format_sql(sql)
        else:
            formatted_sql = sql
        result = LineageRunner(formatted_sql,dialect='non-validating')
        source_tables = [str(i).split('.')[1] for i in result.source_tables]
        return source_tables
    except:
        return []

# def get_sql_from_table_name(table_name):
#     query_data = {
#         "table_name": table_name
#     }
#     response = api.post(query_data)
#     return response[0]['file_content']

def get_sql_from_table_name(table_name):
    ## 在file_path内搜索table_name.sql，返回sql
    file_path = '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/'
    table_name =  table_name+'.sql'
    ## 如果找不到，返回None，否则返回sql
    if table_name not in os.listdir(file_path):
        return None
    with open(file_path+table_name,'r') as f:
        sql = f.read()
    return sql
    

## 给定初始化的sql，获取所有的source tables，返回一个list，继续递归直到无法通过table_name找到sql,
## 把所有的sql都获取到，存在一个list里面，一开始的sql也要加进去
## 再维护一个list，存储已经获取过的table_name，避免重复获取，如果已经获取过，就不再获取
all_sql = []
source_tables = []

def get_all_source_tables(sql,level=3):
    ## 超过5层递归，返回
    if level == 0:
        return
    if sql is None:
        return 
    source_tables_ = get_source_tables_from_sql(sql)
    print(source_tables_)
    for table_name in source_tables_:
        if table_name not in source_tables:
            source_tables.append(table_name)
            sql = get_sql_from_table_name(table_name)
            all_sql.append(sql)
            get_all_source_tables(sql,level-1)
        
            

def combine_sql(sql_list):
    # 去掉None  
    sql_list = [i for i in sql_list if i is not None]
    # print(len(sql_list))
    # 如果最后不是分号结尾，加上分号
    for i in range(len(sql_list)):
        if sql_list[i][-1] != ';':
            sql_list[i] += ';'
    return '\n'.join(sql_list)


all_sql.append(sql)
get_all_source_tables(sql)
print(len(all_sql))

[]
1


/tmp/ipykernel_13483/769870701.py:9: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(formatted_sql,dialect='non-validating')


In [22]:
get_source_tables_from_sql(sql)

/tmp/ipykernel_1630885/769870701.py:9: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(formatted_sql,dialect='non-validating')


['case_order',
 'dwd_rt3_case_base_execution_with_order_hf',
 'dwd_rt3_parent_and_child_order_package_hf',
 'dwd_rt3_task_base_schedule_execution_hf',
 'ods_operation_associate_case_child_order_package']

### 本地表导入

In [13]:
import pandas as pd
import numpy as np


file_path = '/home/zhengzong/workspace/DS/sql任务表查询.txt'
## read txt to df
sql_code_df = pd.read_csv(file_path,sep='\t')

## 根据file_name筛选
def select_sql_from_file_name(df):
    # 过滤.csv / .xlsx / .xls .zip结尾的
    df = df[~df['file_name'].str.contains('.csv|.xlsx|.xls|.zip')]
    # 过滤Hive2ClickHous、Cooper2Hive、Hive2MySQL、
    df = df[~df['file_name'].str.contains('Hive2ClickHous|Cooper2Hive|Hive2MySQL|MySQL2Hive|MysqlToHive|hive2ck')]
    # 过滤掉test的
    df = df[~df['file_name'].str.contains('test|tmp')]
    # 过滤掉query的
    df = df[~df['file_name'].str.contains('query')]
    # 过滤掉空的file_content
    df = df[df['file_content'].notnull()]
    # 过滤掉.txt/.conf结尾的
    df = df[~df['file_name'].str.contains('.txt|.conf')]
    # 过滤掉datalinkapi_开头的
    df = df[~df['file_name'].str.contains('datalinkapi_')]
    # 过滤掉alter开头的
    df = df[~df['file_name'].str.contains('alter')]
    return df

sql_code_df = select_sql_from_file_name(sql_code_df)

In [15]:
sql_code_df

,id,file_id,file_version,file_content,commit_time,commit_user,schedule_content,schedule_uuid,file_type,use_type,file_desc,file_name,status,change_type,is_current_prod,project_id,file_property_content,parent_file_id
3,35846765,18136735,2,--SPARK_SQL_brp_--****************************...,2022-07-26 19:09:22.000,sherlockshen,"{""alarmMode"":"""",""alarmOdinGroup"":"""",""alarmRece...",vgds.dwd_autolabel_tp_lane_change_july,104,2,dwd_autolabel_tp_lane_change_july,dwd_autolabel_tp_lane_change_july,101,1,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
5,35864359,18133447,1,from pyspark.sql.functions import udf_brp_from...,2022-07-26 19:34:50.000,zhonghao_i,NaN,NaN,212,4,NaN,get_latlon_by_pose.py,101,0,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
7,40977787,19959513,6,#!/usr/bin/env python_brp_# -*- coding: utf-8 ...,2022-12-27 17:12:38.000,yanranhan,"{""alarmDchatGroup"":"""",""alarmMode"":"""",""alarmOdi...",vgds.route_grade_odd,105,2,for us,route_grade_odd,101,1,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
8,41027893,20102029,3,#!/usr/bin/env python_brp_# -*- coding: utf-8 ...,2022-12-30 10:43:56.000,jimmylimao,"{""alarmDchatGroup"":"""",""alarmMode"":"""",""alarmOdi...",vgds.ego_lean_one_side_2,105,2,2,ego_lean_one_side_2,101,1,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
9,41037721,14243975,20,--SPARK_SQL_brp_--****************************...,2022-12-30 18:44:18.000,yingyan,"{""alarmDchatGroup"":""[{\""name\"":\""auto labeling...",vgds.dwd_autolabel_junction,104,1,dwd_autolabel_junction,dwd_autolabel_junction,101,1,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,50093889,23495898,13,--@exclude_dependency=vgds.app_rt_trip_issue_d...,2024-02-28 19:12:41.000,haukwen,"{""alarmDchatGroup"":"""",""alarmMode"":"""",""alarmOdi...",vgds.app_rt_master_issue_stat,104,1,计算了master今日issue统计、同比昨天，环比上周,app_rt_master_issue_stat,101,1,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
1791,50095695,25599305,1,--@exclude_dependency=vgds.ods_all_roads_daily...,2024-02-28 20:30:25.000,halldong,"{""alarmDchatGroup"":"""",""alarmMode"":"""",""alarmOdi...",vgds.app_junction_lane_percent_di,104,1,road和lane级别的人/自驾差异指数前置数据,app_junction_lane_percent_di,101,0,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
1794,56645483,29648721,6,--@exclude_dependency=sparklingwater.gateway_d...,2024-07-30 11:21:34.000,jimmylimao,"{""alarmDchatGroup"":"""",""alarmMode"":"""",""alarmOdi...",vgds.app_trip_rt_event_eb_eh,104,1,Stats of EB and EH triggered times.,app_trip_rt_event_eb_eh,101,1,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0
1795,56771577,30791975,2,--SPARK_SQL_brp_--****************************...,2024-08-01 12:31:30.000,taokexin,"{""alarmDchatGroup"":"""",""alarmMode"":"""",""alarmOdi...",vgds.dwd_ra_dotting_waypoint_and_light_assist_...,104,1,-,dwd_ra_dotting_waypoint_and_light_assist_time_...,101,1,1,3100,"{""commitStatus"":0,""countryGroupCode"":"""",""count...",0


In [13]:
sql_code_df[sql_code_df['file_name'] == 'dwd_rt3_task_order_package_case_order_hf']['file_content'].iloc[0]

"--SPARK_SQL_brp_--********************************************************************--_brp_--author:taokexin_brp_--create time:2023-11-07 17:02:15_brp_--desc:区域测试宽表for 路测执行治理_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_-- drop table if exists dwd_rt3_task_order_package_case_order_hf;_brp_create table if not exists dwd_rt3_task_order_package_case_order_hf_brp_(_brp_  `task_base_absolute_priority` bigint COMMENT '绝对优先级'_ _brp_  `task_base_auto_allocation` bigint COMMENT '是否自动分配(0:关闭_ 1:开启）'_ _brp_  `task_base_available_cars` string COMMENT '筛选的可供任务调用的自动驾驶车辆ID列表'_ _brp_  `task_base_available_driver` string COMMENT '筛选的可供任务调用的自动驾驶车驾驶员工号列表'_ _brp_  `task_base_available_follower` string COMMENT '筛选的可供任务调用的保护车驾驶员工号列表'_ _brp_  `task_base_available_follower_cars` string COMMENT '筛选的可供任务调用的保护车ID列表'_ _brp_  `task_base_available_safety` string COMMENT '筛选的可供任务调用的自动驾驶车安全员工号列表'_ _brp_  `task_base_base_task_detail` string COMMENT '

In [3]:
sql = sql_code_df[sql_code_df['file_name'] == 'dim_rt_trip_odd_info']['file_content'].iloc[0]

In [4]:
sql

"--@exclude_dependency=vgds.dim_rt_trip_odd_static_df_brp_--@exclude_dependency=vgds.dim_rt_trip_odd_static_brp_--SPARK_SQL_brp_--********************************************************************--_brp_--author:zhengzong_brp_--create time:2024-09-06 15:18:31_brp_--desc:trip_id与trip_odd的对应表_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_-- DROP TABLE `dim_rt_trip_odd_info_df`;_brp_CREATE TABLE if not exists  `dim_rt_trip_odd_info_df`(_brp_  `trip_id` string COMMENT 'trip_id'_ _brp_  `trip_odd` string COMMENT 'trip_odd'__brp_  `create_date` string COMMENT 'create_date'_brp_  )_brp_PARTITIONED BY ( _brp_  `dt` string)_brp_;_brp__brp_WITH app_rt_trip_issue_detail_hf AS (_brp_    SELECT DISTINCT_brp_        trip_id__brp_        region__brp_        create_date_brp_    FROM _brp_        vgds.app_rt_trip_issue_detail_hf_brp_    WHERE _brp_        dt = '${BIZ_HOUR_LINE}'_brp_)__brp__brp_trip_odd_data AS (_brp_    SELECT DISTINCT

In [16]:
# print(format_sql(sql))

In [17]:
from tqdm import tqdm
import os
import shutil

data_path = '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/'

## 清空data_path下的文件
shutil.rmtree(data_path)
os.makedirs(data_path)


## 遍历sql_code_df,将file_content写入data_path, 并以file_name命名
for index, row in tqdm(sql_code_df.iterrows()):
    file_name = row['file_name'] + '.sql'
    file_content = row['file_content']
    ## 格式化sql
    print(file_name)
    try:
        formatted_sql = format_sql(file_content)
    except:
        print(file_name, '格式化失败')
        print(file_content)
        continue
    
    with open(data_path + file_name, 'w') as f:
        f.write(formatted_sql)

0it [00:00, ?it/s]

1068it [00:00, 5423.82it/s]

dwd_autolabel_tp_lane_change_july.sql
get_latlon_by_pose.py.sql
route_grade_odd.sql
ego_lean_one_side_2.sql
dwd_autolabel_junction.sql
dwd_weather_district.sql
dwm_autolabel_labeled_detail.sql
app_temp_dwm_car_eff.sql
pull_status_calculate.sql
app_rt_followcar_coverage_inserthistorydata.sql
app_rt_followcar_coverage_df.sql
dm_scenario_lane_change_d.sql
dwd_asm_ego_obj_contour.sql
app_autotopic_issue_diff_detail.sql
app_ds_autotopic_accuracy.sql
dwd_release_binary.sql
dwm_voyager_car_group_on_off_line.sql
app_ds_get_latlon_by_pose.sql
holiday_for_ops.sql
dwd_autolabel_road_info.sql
dwd_sim_datasim_scenario_detail.sql
app_dwd_car_eff_temp.sql
ops_team_leader.sql
app_dwd_people_eff_temp.sql
app_dwm_car_eff_temp.sql
dim_ds_issue_pose_di.sql
dim_ds_issue_speed_di.sql
dwd_ds_mdbi_issue_new_di.sql
app_ds_mdbi_trip_issue_v4_di.sql
dwd_autolabel_rule_lane_change.sql
ce_issue_cretieria_to_hive.sql
dwd_voyager_opstrain.sql
dwd_ds_trip_exemption_st_di.sql
app_ds_mdbi_trip_issue_v3_di.sql
dwd_rtmap

In [8]:
table_name = 'dwd_rt3_task_order_package_case_order_hf'
